# NB16 — chemCPA Scaffold Parity Run (Corrected)

This notebook is a **best-effort Colab parity run** for `chemCPA` on the **same dataset + same scaffold split** used in NB10–NB15.

It is designed to be honest:

- if training truly starts, the notebook should produce logs / checkpoints / markers
- if something fails, the notebook should fail loudly or leave a usable log
- it does **not** fake parity

This notebook keeps:
- **repo in `/content/chemCPA`**
- **data + final results in Google Drive**
- **temporary logs/work dirs in `/content/...`**

## Important note

This notebook tries a Colab-compatible route.  
If the official repo expects a stricter conda/Docker environment, the run may still fail.

That is acceptable.  
The goal here is to get a **real chemCPA parity attempt**, not a fake comparison.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
# If an older broken install already ran, restart runtime once before continuing.

!pip -q uninstall -y pandas
!pip -q install --no-cache-dir     pandas==2.2.2     anndata==0.10.8     scanpy==1.10.2     scipy==1.13.1     scikit-learn==1.5.2     matplotlib==3.10.0     seaborn==0.13.2     h5py pyarrow fastparquet     lightning==2.4.0     torchmetrics==1.4.1     hydra-core==1.3.2     submitit seml     rdkit     deepchem     dgllife     jupytext

In [ ]:
import pandas as pd
print("pandas version:", pd.__version__)
assert pd.__version__ == "2.2.2"

In [ ]:
import os
import json
import yaml
import glob
import shutil
import warnings
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import anndata as ad

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)

In [ ]:
@dataclass
class CFG:
    # core data
    data_path: str = "/content/drive/MyDrive/ChemDFM/data/sciplex_complete_middle_subset.h5ad"

    # NB10 artifacts
    nb10_res_dir: str = "/content/drive/MyDrive/ChemDFM/results_nb10_scaffold"
    nb10_art_dir: str = "/content/drive/MyDrive/ChemDFM/results_nb10_scaffold/artifacts"

    # outputs
    results_dir: str = "/content/drive/MyDrive/ChemDFM/results_chemcpa_scaffold"
    art_dir: str = "/content/drive/MyDrive/ChemDFM/results_chemcpa_scaffold/artifacts"

    # repo
    repo_root: str = "/content/chemCPA"
    repo_url: str = "https://github.com/theislab/chemCPA.git"

    # derived dataset
    derived_h5ad: str = "/content/drive/MyDrive/ChemDFM/results_chemcpa_scaffold/sciplex_scaffold_for_chemcpa.h5ad"

    # schema
    split_col: str = "split_scaffold"
    condition_col: str = "condition"
    cell_col: str = "cell_type"
    dose_col: str = "dose"
    fallback_dose_col: str = "dose_val"
    control_label: str = "control"
    smiles_candidates: tuple = ("SMILES", "canonical_smiles", "smiles", "Smiles", "smiles_rdkit", "structure")

    # chemCPA-friendly fields
    smiles_out_col: str = "SMILES"
    pert_category_col: str = "cov_drug_dose_name"

    # run config
    embedding_model: str = "rdkit"
    num_epochs: int = 60
    checkpoint_freq: int = 20
    max_minutes: int = 240
    batch_size: int = 128
    random_seed: int = 42

    # optional pretrained toggle
    load_pretrained: bool = False
    pretrained_model_path: str = ""
    pretrained_rdkit_hash: str = "4f061dbfc7af05cf84f06a724b0c8563"

cfg = CFG()
Path(cfg.results_dir).mkdir(parents=True, exist_ok=True)
Path(cfg.art_dir).mkdir(parents=True, exist_ok=True)

print("NB16 config:")
for k, v in asdict(cfg).items():
    print(f"  {k}: {v}")

## Helper utilities

In [ ]:
def read_json(path):
    if path is None or not os.path.exists(path):
        return None
    with open(path, "r") as f:
        return json.load(f)

def read_csv(path):
    if path is None or not os.path.exists(path):
        return None
    return pd.read_csv(path)

def choose_first_existing(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None

def first_existing_path(paths):
    for p in paths:
        if p is not None and os.path.exists(p):
            return p
    return None

def safe_glob(pattern):
    return sorted(glob.glob(pattern, recursive=True))

## Recover scaffold split and build chemCPA-ready h5ad

In [ ]:
adata = ad.read_h5ad(cfg.data_path)
if cfg.dose_col not in adata.obs.columns and cfg.fallback_dose_col in adata.obs.columns:
    adata.obs[cfg.dose_col] = adata.obs[cfg.fallback_dose_col]

nb10_gate_path = first_existing_path([
    os.path.join(cfg.nb10_res_dir, "scaffold_split_gate.json"),
    os.path.join(cfg.nb10_art_dir, "scaffold_split_gate.json"),
])
scaffold_map_path = first_existing_path([
    os.path.join(cfg.nb10_res_dir, "scaffold_split_map.csv"),
    os.path.join(cfg.nb10_art_dir, "scaffold_split_map.csv"),
])
obs_nb10_path = first_existing_path([
    os.path.join(cfg.nb10_res_dir, "obs_with_scaffold_split.csv"),
    os.path.join(cfg.nb10_art_dir, "obs_with_scaffold_split.csv"),
    os.path.join(cfg.nb10_res_dir, "adata_obs_with_scaffold_split.csv"),
    os.path.join(cfg.nb10_art_dir, "adata_obs_with_scaffold_split.csv"),
])

nb10_gate = read_json(nb10_gate_path)
scaffold_map = read_csv(scaffold_map_path)
obs_nb10 = read_csv(obs_nb10_path)

if cfg.split_col not in adata.obs.columns:
    recovered = False

    if obs_nb10 is not None:
        idx_col = choose_first_existing(list(obs_nb10.columns), ["obs_index", "index", "Unnamed: 0"])
        if idx_col is not None and cfg.split_col in obs_nb10.columns:
            temp = obs_nb10[[idx_col, cfg.split_col]].copy()
            temp = temp.rename(columns={idx_col: "_obs_idx_nb16"})
            temp["_obs_idx_nb16"] = temp["_obs_idx_nb16"].astype(str)
            adata.obs["_obs_idx_nb16"] = adata.obs.index.astype(str)
            adata.obs = adata.obs.merge(temp, on="_obs_idx_nb16", how="left")
            adata.obs.index = adata.obs["_obs_idx_nb16"].astype(str)
            if not adata.obs[cfg.split_col].isna().any():
                recovered = True
                print(f"Recovered {cfg.split_col} from NB10 obs table.")

    if not recovered and scaffold_map is not None:
        if cfg.split_col not in scaffold_map.columns:
            raise RuntimeError(f"scaffold_map lacks {cfg.split_col}")
        drug_col = choose_first_existing(
            list(scaffold_map.columns),
            [cfg.condition_col, "drug", "drug_name", "condition", "compound", "perturbation"]
        )
        if drug_col is None:
            raise RuntimeError(f"Could not find drug column in scaffold_map: {list(scaffold_map.columns)}")
        mapper = scaffold_map[[drug_col, cfg.split_col]].drop_duplicates().set_index(drug_col)[cfg.split_col]
        cond_series = adata.obs[cfg.condition_col].astype(str)
        adata.obs[cfg.split_col] = np.where(
            cond_series == cfg.control_label,
            cfg.control_label,
            cond_series.map(mapper)
        )
        missing_mask = adata.obs[cfg.split_col].isna() & (cond_series != cfg.control_label)
        if missing_mask.any():
            missing_drugs = sorted(cond_series[missing_mask].unique().tolist())[:20]
            raise RuntimeError(f"Some perturbed drugs still missing split labels. Examples: {missing_drugs}")
        recovered = True
        print(f"Recovered {cfg.split_col} from scaffold_map.")

    if not recovered:
        raise RuntimeError(f"Could not recover {cfg.split_col} from NB10 artifacts.")

required_cols = [cfg.condition_col, cfg.cell_col, cfg.dose_col, cfg.split_col]
missing = [c for c in required_cols if c not in adata.obs.columns]
if missing:
    raise RuntimeError(f"Missing required columns after recovery: {missing}")

smiles_col = choose_first_existing(list(adata.obs.columns), cfg.smiles_candidates)
if smiles_col is None:
    raise RuntimeError(
        f"No usable SMILES column found. Tried: {cfg.smiles_candidates}. "
        f"Available obs columns: {list(adata.obs.columns)}"
    )

if cfg.smiles_out_col not in adata.obs.columns:
    adata.obs[cfg.smiles_out_col] = adata.obs[smiles_col].astype(str)
else:
    adata.obs[cfg.smiles_out_col] = adata.obs[cfg.smiles_out_col].astype(str)

if cfg.pert_category_col not in adata.obs.columns:
    adata.obs[cfg.pert_category_col] = (
        adata.obs[cfg.cell_col].astype(str) + "_" +
        adata.obs[cfg.condition_col].astype(str) + "_" +
        adata.obs[cfg.dose_col].astype(str)
    )

print("Original split label counts:")
display(
    adata.obs[cfg.split_col].astype(str).value_counts(dropna=False).rename_axis("split").reset_index(name="n")
)

# training-facing split copy for chemCPA
adata.obs["split_scaffold_chemcpa"] = adata.obs[cfg.split_col].astype(str)
adata.obs.loc[adata.obs["split_scaffold_chemcpa"] == cfg.control_label, "split_scaffold_chemcpa"] = "train"

print("chemCPA-facing split label counts:")
display(
    adata.obs["split_scaffold_chemcpa"].astype(str).value_counts(dropna=False).rename_axis("split").reset_index(name="n")
)

audit = {
    "n_cells": int(adata.n_obs),
    "n_genes": int(adata.n_vars),
    "smiles_col_source": smiles_col,
    "final_smiles_col": cfg.smiles_out_col,
    "split_col_original": cfg.split_col,
    "split_col_for_chemcpa": "split_scaffold_chemcpa",
    "load_pretrained": bool(cfg.load_pretrained),
}
with open(os.path.join(cfg.results_dir, "dataset_patch_audit_nb16.json"), "w") as f:
    json.dump(audit, f, indent=2)

adata.write_h5ad(cfg.derived_h5ad)
print("Wrote derived chemCPA-ready h5ad to:", cfg.derived_h5ad)

## Clone repo and install editable package

In [ ]:
if os.path.exists(cfg.repo_root):
    print("Repo already exists, pulling latest main...")
    !git -C {cfg.repo_root} fetch --all
    !git -C {cfg.repo_root} checkout main
    !git -C {cfg.repo_root} pull
else:
    !git clone {cfg.repo_url} {cfg.repo_root}

print("Repo ready at:", cfg.repo_root)
!ls -la {cfg.repo_root}

In [ ]:
%cd /content/chemCPA
!pip -q install -e .
%cd /content

## Write corrected local-writable chemCPA config

In [ ]:
LOCAL_LOG_DIR = "/content/chemCPA_local_logs"
LOCAL_WORK_DIR = "/content/chemCPA_local_work"
Path(LOCAL_LOG_DIR).mkdir(parents=True, exist_ok=True)
Path(LOCAL_WORK_DIR).mkdir(parents=True, exist_ok=True)
Path(os.path.join(cfg.results_dir, "checkpoints")).mkdir(parents=True, exist_ok=True)

manual_cfg = {
    "seml": {
        "executable": "chemCPA/experiments_run.py",
        "name": "scaffold_parity",
        "output_dir": LOCAL_LOG_DIR,
        "conda_environment": "chemCPA",
        "project_root_dir": cfg.repo_root,
    },
    "slurm": {
        "max_simultaneous_jobs": 1,
        "experiments_per_job": 1,
        "sbatch_options_template": "GPU",
        "sbatch_options": {
            "gres": "gpu:1",
            "mem": "32G",
            "cpus-per-task": 4,
            "time": "0-12:00",
            "nice": 10000,
        },
    },
    "fixed": {
        "profiling.run_profiler": False,
        "profiling.outdir": LOCAL_WORK_DIR,

        "training.checkpoint_freq": cfg.checkpoint_freq,
        "training.num_epochs": cfg.num_epochs,
        "training.max_minutes": cfg.max_minutes,
        "training.full_eval_during_train": False,
        "training.run_eval_disentangle": True,
        "training.run_eval_r2": True,
        "training.run_eval_r2_sc": False,
        "training.run_eval_logfold": False,
        "training.save_checkpoints": True,
        "training.save_dir": str(Path(cfg.results_dir) / "checkpoints"),

        "dataset.dataset_type": "trapnell",
        "dataset.data_params.dataset_path": cfg.derived_h5ad,
        "dataset.data_params.perturbation_key": cfg.condition_col,
        "dataset.data_params.pert_category": cfg.pert_category_col,
        "dataset.data_params.dose_key": cfg.dose_col,
        "dataset.data_params.covariate_keys": cfg.cell_col,
        "dataset.data_params.smiles_key": cfg.smiles_out_col,
        "dataset.data_params.use_drugs_idx": True,
        "dataset.data_params.split_key": "split_scaffold_chemcpa",
        "dataset.data_params.degs_key": "all_DEGs",

        "model.enable_cpa_mode": False,
        "model.embedding.directory": None,
        "model.embedding.model": cfg.embedding_model,
        "model.load_pretrained": bool(cfg.load_pretrained),
        "model.pretrained_model_path": cfg.pretrained_model_path if cfg.load_pretrained else "",
        "model.pretrained_model_hashes": {
            "rdkit": cfg.pretrained_rdkit_hash
        },

        "model.additional_params.seed": cfg.random_seed,
        "model.additional_params.patience": 30,
        "model.additional_params.decoder_activation": "ReLU",
        "model.additional_params.doser_type": "amortized",

        "model.hparams.dim": 32,
        "model.hparams.dropout": 0.262378,
        "model.hparams.autoencoder_width": 256,
        "model.hparams.autoencoder_depth": 4,
        "model.hparams.batch_size": cfg.batch_size,
        "model.hparams.reg_multi_task": 0,
        "model.hparams.dosers_width": 64,
        "model.hparams.dosers_depth": 3,
        "model.hparams.step_size_lr": 50,
        "model.hparams.embedding_encoder_width": 128,
        "model.hparams.embedding_encoder_depth": 4,
        "model.append_ae_layer": True,
    },
    "random": {
        "samples": 1,
        "seed": cfg.random_seed,
        "model.hparams.autoencoder_lr": {"type": "choice", "options": [1e-3]},
        "model.hparams.autoencoder_wd": {"type": "choice", "options": [1e-6]},
        "model.hparams.adversary_width": {"type": "choice", "options": [128]},
        "model.hparams.adversary_depth": {"type": "choice", "options": [3]},
        "model.hparams.adversary_lr": {"type": "choice", "options": [1e-3]},
        "model.hparams.adversary_wd": {"type": "choice", "options": [1e-5]},
        "model.hparams.adversary_steps": {"type": "choice", "options": [2]},
        "model.hparams.reg_adversary": {"type": "choice", "options": [24.082073]},
        "model.hparams.reg_adversary_cov": {"type": "choice", "options": [10.0]},
        "model.hparams.penalty_adversary": {"type": "choice", "options": [1.0]},
        "model.hparams.dosers_lr": {"type": "choice", "options": [1e-3]},
        "model.hparams.dosers_wd": {"type": "choice", "options": [1e-6]},
    },
    "grid": {}
}

manual_yaml_path = os.path.join(cfg.repo_root, "manual_scaffold_parity.yaml")
with open(manual_yaml_path, "w") as f:
    yaml.safe_dump(manual_cfg, f, sort_keys=False)

print("Wrote config to:", manual_yaml_path)
print("Local log dir:", LOCAL_LOG_DIR)
print("Local work dir:", LOCAL_WORK_DIR)

## Write corrected runner with explicit success/failure markers

In [ ]:
runner_py = r'''
import os
import sys
import json
import traceback
from pathlib import Path
from pprint import pprint

from seml.config import generate_configs, read_config
from chemCPA.experiments_run import ExperimentWrapper

CONFIG = "manual_scaffold_parity.yaml"

def main():
    print("=" * 80, flush=True)
    print("chemCPA scaffold parity runner starting", flush=True)
    print("CWD:", os.getcwd(), flush=True)
    print("CONFIG:", CONFIG, flush=True)
    print("=" * 80, flush=True)

    assert Path(CONFIG).exists(), f"config file not found: {CONFIG}"

    seml_config, slurm_config, experiment_config = read_config(CONFIG)
    configs = generate_configs(experiment_config)
    print(f"Generated configs: {len(configs)}", flush=True)
    if len(configs) > 1:
        print("Warning: more than one config generated; using the first one.", flush=True)

    args = configs[0]
    pprint(args)

    exp = ExperimentWrapper(init_all=False)
    exp.seed = 1337

    print("\n[1/5] init_dataset", flush=True)
    exp.init_dataset(**args["dataset"])
    print("init_dataset done", flush=True)

    print("\n[2/5] init_drug_embedding", flush=True)
    exp.init_drug_embedding(embedding=args["model"]["embedding"])
    print("init_drug_embedding done", flush=True)

    print("\n[3/5] init_model", flush=True)
    exp.init_model(
        hparams=args["model"]["hparams"],
        additional_params=args["model"]["additional_params"],
        load_pretrained=args["model"]["load_pretrained"],
        append_ae_layer=args["model"]["append_ae_layer"],
        enable_cpa_mode=args["model"]["enable_cpa_mode"],
        pretrained_model_path=args["model"]["pretrained_model_path"],
        pretrained_model_hashes=args["model"]["pretrained_model_hashes"],
    )
    print("init_model done", flush=True)

    print("\n[4/5] update_datasets", flush=True)
    exp.update_datasets()
    print("update_datasets done", flush=True)

    print("\n[5/5] train", flush=True)
    result = exp.train(**args["training"])
    print("train done", flush=True)
    print("train result type:", type(result), flush=True)
    print("train result:", result, flush=True)

    marker = {
        "status": "finished_train_call",
        "config_path": str(Path(CONFIG).resolve()),
    }
    with open("nb16_runner_success.json", "w") as f:
        json.dump(marker, f, indent=2)

if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        print("\nFATAL ERROR IN RUNNER", flush=True)
        print(type(e).__name__, str(e), flush=True)
        traceback.print_exc()
        with open("nb16_runner_failure.txt", "w") as f:
            f.write(type(e).__name__ + ": " + str(e) + "\n\n")
            f.write(traceback.format_exc())
        raise
'''
runner_path = os.path.join(cfg.repo_root, "run_scaffold_parity.py")
with open(runner_path, "w") as f:
    f.write(runner_py)

print("Wrote runner to:", runner_path)

## Launch run

In [ ]:
%cd /content/chemCPA

run_log = os.path.join(cfg.results_dir, "chemcpa_run_console.log")
if os.path.exists(run_log):
    os.remove(run_log)

cmd = f"python -u run_scaffold_parity.py 2>&1 | tee {run_log}"
print("Running:", cmd)
ret = os.system(cmd)
print("Return code:", ret)

print("\n--- success marker exists? ---")
print(os.path.exists("/content/chemCPA/nb16_runner_success.json"))

print("\n--- failure marker exists? ---")
print(os.path.exists("/content/chemCPA/nb16_runner_failure.txt"))
if os.path.exists("/content/chemCPA/nb16_runner_failure.txt"):
    with open("/content/chemCPA/nb16_runner_failure.txt", "r") as f:
        print(f.read()[:5000])

%cd /content

## Tail log

In [ ]:
print("=== tail of run log ===")
run_log = os.path.join(cfg.results_dir, "chemcpa_run_console.log")
if os.path.exists(run_log):
    with open(run_log, "r") as f:
        lines = f.readlines()
    print("".join(lines[-200:]))
else:
    print("run log not found")

## Scan artifacts from all likely writable locations

In [ ]:
LOCAL_LOG_DIR = "/content/chemCPA_local_logs"
LOCAL_WORK_DIR = "/content/chemCPA_local_work"

search_roots = [
    os.path.join(cfg.repo_root, "**", "*"),
    os.path.join(LOCAL_LOG_DIR, "**", "*"),
    os.path.join(LOCAL_WORK_DIR, "**", "*"),
    os.path.join(cfg.results_dir, "checkpoints", "**", "*"),
]

all_files = []
for pat in search_roots:
    all_files.extend(safe_glob(pat))

all_files = sorted(set([p for p in all_files if os.path.isfile(p)]))

jsons = [p for p in all_files if p.endswith(".json")]
csvs = [p for p in all_files if p.endswith(".csv")]
pt_files = [p for p in all_files if p.endswith(".pt") or p.endswith(".ckpt")]

artifact_inventory = pd.DataFrame({"path": all_files})
artifact_inventory["ext"] = artifact_inventory["path"].map(lambda p: Path(p).suffix)
display(artifact_inventory.head(100))

artifact_inventory.to_csv(
    os.path.join(cfg.results_dir, "chemcpa_artifact_inventory.csv"),
    index=False
)

print("Found:")
print("  files:", len(all_files))
print("  json :", len(jsons))
print("  csv  :", len(csvs))
print("  ckpt :", len(pt_files))

## Try automatic parity extraction

In [ ]:
candidate_metrics = []
for p in csvs:
    try:
        df = pd.read_csv(p)
    except Exception:
        continue
    cols = set(map(str, df.columns))
    if "r2_top50" in cols or "r2" in cols or "pearson_row" in cols:
        candidate_metrics.append((p, df))

print("Candidate metric files:", len(candidate_metrics))
for p, df in candidate_metrics[:10]:
    print("\n---", p)
    display(df.head())

exported = False
for p, df in candidate_metrics:
    cols = set(map(str, df.columns))
    if {"method", "split", "r2_top50"}.issubset(cols):
        out = df.copy()
        out.to_csv(os.path.join(cfg.results_dir, "chemcpa_scaffold_metrics.csv"), index=False)
        exported = True
        print("Exported direct parity CSV from:", p)
        break

if not exported:
    print("No directly usable same-schema metrics CSV was found.")

## Manual parity template

Use this only if training succeeded but auto extraction failed.

In [ ]:
manual_metrics_path = os.path.join(cfg.results_dir, "chemcpa_scaffold_metrics.csv")

manual_template = pd.DataFrame([
    {
        "method": "chemcpa",
        "split": "test",
        "r2_top50": np.nan,
        "r2_top20": np.nan,
        "pearson_row": np.nan,
        "r2_full": np.nan,
        "mse": np.nan,
        "n_cells": np.nan,
    },
    {
        "method": "chemcpa",
        "split": "ood",
        "r2_top50": np.nan,
        "r2_top20": np.nan,
        "pearson_row": np.nan,
        "r2_full": np.nan,
        "mse": np.nan,
        "n_cells": np.nan,
    },
])

display(manual_template)
print("If auto extraction failed but you have final chemCPA metrics, save them to:")
print(manual_metrics_path)

## Final audit

In [ ]:
parity_csv = os.path.join(cfg.results_dir, "chemcpa_scaffold_metrics.csv")
run_log = os.path.join(cfg.results_dir, "chemcpa_run_console.log")
success_marker = "/content/chemCPA/nb16_runner_success.json"
failure_marker = "/content/chemCPA/nb16_runner_failure.txt"

status = {
    "derived_h5ad_exists": os.path.exists(cfg.derived_h5ad),
    "run_log_exists": os.path.exists(run_log),
    "artifact_inventory_exists": os.path.exists(os.path.join(cfg.results_dir, "chemcpa_artifact_inventory.csv")),
    "success_marker_exists": os.path.exists(success_marker),
    "failure_marker_exists": os.path.exists(failure_marker),
    "parity_csv_exists": os.path.exists(parity_csv),
}

status_df = pd.DataFrame([status]).T.reset_index()
status_df.columns = ["field", "value"]
display(status_df)
status_df.to_csv(os.path.join(cfg.results_dir, "nb16_final_status.csv"), index=False)

if status["parity_csv_exists"]:
    print("SUCCESS: chemCPA parity CSV exists.")
    print(parity_csv)
elif status["success_marker_exists"]:
    print("PARTIAL: train call finished, but parity CSV was not auto-extracted.")
    print("Check log and artifact inventory.")
elif status["failure_marker_exists"]:
    print("FAIL: runner raised an exception.")
    with open(failure_marker, "r") as f:
        print(f.read()[:5000])
else:
    print("UNKNOWN: no parity CSV and no explicit success/failure marker.")
    print("Check the run log tail carefully.")